In [2]:
from dataclasses import dataclass
from typing import List, Dict

In [3]:
@dataclass
class ChoiceDecision:
  question: str
  options: List[str]

@dataclass
class BooleanDecision:
  question: str

In [4]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

In [7]:
class ZeroTokenDecisionEngine(nn.Module):
  def __init__(self, model_name: str = 'google/electra-base-discriminator'):
    super().__init__()
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.encoder = AutoModel.from_pretrained(model_name)
    hidden_dim = self.encoder.config.hidden_size

    # binary calibration head(returns logit for P(true))
    self.binary_head = nn.Sequential(
      nn.Linear(hidden_dim, hidden_dim // 2),
      nn.GELU(),
      nn.Dropout(0.1),
      nn.Linear(hidden_dim // 2, 1)
    )

    # projection head for dynamic zero-shot semantic matching
    self.similarity_proj = nn.Linear(hidden_dim, hidden_dim)

  def _get_context_embedding(self, text: str) -> torch.Tensor:
    """extract the pooled representation of the state/question in one forward pass"""
    encoded = self.tokenizer(
        text,
        return_tensors= "pt",
        truncation = True,
        max_length = 512,
        padding = True
    )
    outputs = self.encoder(**encoded)
    # use [CLS] token representation
    return outputs.last_hidden_state[:, 0, :]

  def evaluate_boolean(self, context: str, question: str) -> float:
    """
    Evaluates a binary statement in a single forward pass without emitting tokens.
    eg: "does this message request account deletion?" -> P(True)
    """

    input_text = f"Context: {context} | Question: {question}"
    h = self._get_context_embedding(input_text)
    logit = self.binary_head(h)
    prob = torch.sigmoid(logit).item()
    return round(prob, 4)

  def evaluate_choice(self, context: str, decision: ChoiceDecision) -> Dict[str, float]:
    """
    calculates exact softmax probabilities over all options simultaneously
    -> no autoregression, no json generation
    """
    # 1. embed context + question
    query_text = f"Context: {context} | Task: {decision.question}"
    query_emb = self.similarity_proj(self._get_context_embedding(query_text))

    # 2. embed all candidate choices in a single batched forward pass
    candidate_encoded = self.tokenizer(
        decision.options,
        return_tensors= "pt",
        truncation = True,
        # max_length = 512,
        padding = True
    )

    with torch.no_grad():
      cand_outputs = self.encoder(**candidate_encoded)
      cand_embs = self.similarity_proj(cand_outputs.last_hidden_state[:, 0, :])

    # 3. compute dot-product logits between query and all candidate targets
    logits = torch.matmul(query_emb, cand_embs.T).squeeze(0)

    # 4. normalize to valid probability distribution
    probs = torch.softmax(logits, dim=-1).tolist()

    return {opt: round(prob, 4) for opt, prob in zip(decision.options, probs)}

In [8]:
# initialize engine
engine = ZeroTokenDecisionEngine()
engine.eval()

# sample agent state
user_query = "Can you refund my subscription for last month? I was charged twice."
current_state = f"User message: '{user_query}'. User status: Active paid tier."

# Decision 1: Boolean Policy Check
is_escalation_needed = engine.evaluate_boolean(
    context=current_state,
    question="Does the user request financial compensation or refund?"
)
print(f"Policy Check - Needs Refund Handling: {is_escalation_needed * 100:.1f}%")

# Decision 2: Multi-class Tool Router (Choice)
routing_decision = ChoiceDecision(
    question="Select the optimal tool to handle this step",
    options=[
        "search_documentation",
        "fetch_billing_ledger",
        "reset_password_workflow",
        "handoff_to_human_agent"
    ]
)

probabilities = engine.evaluate_choice(current_state, routing_decision)
print("\nTool Routing Probabilities:")
for tool, p in sorted(probabilities.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {tool:25s}: {p*100:5.1f}%")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Policy Check - Needs Refund Handling: 51.1%

Tool Routing Probabilities:
  - handoff_to_human_agent   :  56.9%
  - reset_password_workflow  :  42.4%
  - search_documentation     :   0.6%
  - fetch_billing_ledger     :   0.0%
